<h1> Setting Up

In [293]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os
from sqlalchemy import create_engine

<h1> Import Dataset

In [294]:
df = pd.read_csv('scraped_data.csv')

In [295]:
display(df.head())
display(df.info())

,URL,Name,Location,Price,Ratings,Check_In_Date,Scraped_At,Search_URL
0,https://www.booking.com/hotel/gb/brit-hotels-e...,Brit Hotels Earls Court,"Kensington and Chelsea, London",US$269,Scored 8.3\n8.3\nVery Good\n734 reviews,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.017755,https://www.booking.com/searchresults.html?ss=...
1,https://www.booking.com/hotel/gb/lancaster-gat...,Lancaster Gate Hotel,"Westminster Borough, London",US$303,"Scored 7.2\n7.2\nGood\n9,222 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.200365,https://www.booking.com/searchresults.html?ss=...
2,https://www.booking.com/hotel/gb/riu-plaza-lon...,Riu Plaza London Victoria,"Westminster Borough, London",US$553,"Scored 8.6\n8.6\nExcellent\n15,378 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.302497,https://www.booking.com/searchresults.html?ss=...
3,https://www.booking.com/hotel/gb/ibis-london-c...,ibis London City - Shoreditch,"Tower Hamlets, London",US$598,"Scored 8.1\n8.1\nVery Good\n6,553 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.393239,https://www.booking.com/searchresults.html?ss=...
4,https://www.booking.com/hotel/gb/sunborn-londo...,Sunborn London Yacht Hotel,"Newham, London",US$399,"Scored 8.8\n8.8\nExcellent\n6,053 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.493018,https://www.booking.com/searchresults.html?ss=...


<class 'pandas.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   URL            643 non-null    str  
 1   Name           643 non-null    str  
 2   Location       643 non-null    str  
 3   Price          643 non-null    str  
 4   Ratings        637 non-null    str  
 5   Check_In_Date  643 non-null    str  
 6   Scraped_At     643 non-null    str  
 7   Search_URL     643 non-null    str  
dtypes: str(8)
memory usage: 40.3 KB


None

<h1> Data Cleaning

In [296]:
df_cleaned = df.copy()

df_cleaned = df_cleaned.drop_duplicates(keep='last')

In [297]:
# Rename Columns to Lower Case
df_cleaned.columns = df_cleaned.columns.str.lower()

In [298]:
# Extract URL to Get Property ID
df_cleaned['property_id'] = df_cleaned['url'].str.extract(r'hotel/[^/]+/([^.]+)\.html')

# Extract Locations into Multiple Columns
df_cleaned[['borough', 'city']] = df_cleaned['location'].str.split(', ', expand=True)
# For 'Lodon' only cases: Move Lodon from borough to city
mask = df_cleaned['city'].isna()
df_cleaned.loc[mask, 'city'] = df_cleaned.loc[mask, 'borough'] 
df_cleaned.loc[mask, 'borough'] = np.nan

# Extract Ratings into Multiple Columns
df_cleaned[['score_text', 'score', 'rating', 'reviews']] = df_cleaned['ratings'].str.split('\n', expand=True)

In [299]:
# Convert price, score, reviews to Numeric
# price: also remove non-numeric and ,
df_cleaned['price'] = df_cleaned['price'].str.replace(r'[^\d.]', '', regex=True).str.strip().astype(int)

df_cleaned['score'] = df_cleaned['score'].astype(float)

# reviews: also remove non-numeric and ,
df_cleaned['reviews'] = df_cleaned['reviews'].str.replace(r'[^\d.]', '', regex=True).str.strip().astype('Int64')

In [300]:
# Categorize Ratings with Score < 7
score_mask = df_cleaned['score'] < 7
df_cleaned.loc[score_mask, 'rating'] = pd.cut(df_cleaned.loc[score_mask, 'score'], bins=[0, 2.99, 4.99, 6.99], labels=['Very Poor', 'Poor', 'Average'])

In [301]:
# Fill NULL Values
df_cleaned['reviews'] =  df_cleaned['reviews'].fillna(0)
df_cleaned['score'] =  df_cleaned['score'].fillna(0)
df_cleaned['rating'] =  df_cleaned['rating'].fillna('No Reviews')
df_cleaned['borough'] =  df_cleaned['borough'].fillna('Unknown')

In [302]:
# Drop Columns
df_cleaned = df_cleaned.drop(columns=['ratings', 'score_text', 'location'])

In [303]:
display(df_cleaned.head())
display(df_cleaned.info())
display(df_cleaned.describe(include='all'))

,url,name,price,check_in_date,scraped_at,search_url,property_id,borough,city,score,rating,reviews
0,https://www.booking.com/hotel/gb/brit-hotels-e...,Brit Hotels Earls Court,269,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.017755,https://www.booking.com/searchresults.html?ss=...,brit-hotels-earls-court,Kensington and Chelsea,London,8.3,Very Good,734
1,https://www.booking.com/hotel/gb/lancaster-gat...,Lancaster Gate Hotel,303,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.200365,https://www.booking.com/searchresults.html?ss=...,lancaster-gate,Westminster Borough,London,7.2,Good,9222
2,https://www.booking.com/hotel/gb/riu-plaza-lon...,Riu Plaza London Victoria,553,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.302497,https://www.booking.com/searchresults.html?ss=...,riu-plaza-london-victoria,Westminster Borough,London,8.6,Excellent,15378
3,https://www.booking.com/hotel/gb/ibis-london-c...,ibis London City - Shoreditch,598,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.393239,https://www.booking.com/searchresults.html?ss=...,ibis-london-city,Tower Hamlets,London,8.1,Very Good,6553
4,https://www.booking.com/hotel/gb/sunborn-londo...,Sunborn London Yacht Hotel,399,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.493018,https://www.booking.com/searchresults.html?ss=...,sunborn-london,Newham,London,8.8,Excellent,6053


<class 'pandas.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   url            643 non-null    str    
 1   name           643 non-null    str    
 2   price          643 non-null    int64  
 3   check_in_date  643 non-null    str    
 4   scraped_at     643 non-null    str    
 5   search_url     643 non-null    str    
 6   property_id    643 non-null    str    
 7   borough        643 non-null    str    
 8   city           643 non-null    str    
 9   score          643 non-null    float64
 10  rating         643 non-null    str    
 11  reviews        643 non-null    Int64  
dtypes: Int64(1), float64(1), int64(1), str(9)
memory usage: 61.0 KB


None

,url,name,price,check_in_date,scraped_at,search_url,property_id,borough,city,score,rating,reviews
count,643,643,643.000000,643,643,643,643,643,643,643.000000,643,643.0
unique,643,643,NaN,1,643,1,643,23,1,NaN,8,<NA>
top,https://www.booking.com/hotel/gb/brit-hotels-e...,Brit Hotels Earls Court,NaN,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.017755,https://www.booking.com/searchresults.html?ss=...,brit-hotels-earls-court,Westminster Borough,London,NaN,Good,<NA>
freq,1,1,NaN,643,1,643,1,185,643,NaN,199,<NA>
mean,NaN,NaN,301.984448,NaN,NaN,NaN,NaN,NaN,NaN,7.741213,NaN,1902.975117
std,NaN,NaN,213.340739,NaN,NaN,NaN,NaN,NaN,NaN,1.245514,NaN,3152.431026
min,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.0
25%,NaN,NaN,144.000000,NaN,NaN,NaN,NaN,NaN,NaN,7.200000,NaN,98.5
50%,NaN,NaN,245.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.000000,NaN,796.0
75%,NaN,NaN,390.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.500000,NaN,2347.5


<h1> Export Dataset

In [307]:
# Export to CSV
file_name = 'cleaned_data.csv'
file_exists = os.path.isfile(file_name)
df_cleaned.to_csv(file_name, index=False, mode='a', header= not file_exists)

In [ ]:
# Append to Database
server = r'(localdb)\MSSQLLocalDB'
database = 'da_projects'
driver = 'ODBC Driver 18 for SQL Server'

connection_string = rf"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
engine = create_engine(connection_string, fast_executemany=True)

# Load Dataframe into Database
table_name = 'bookings'
df_cleaned.to_sql(
    name=table_name,
    con=engine, 
    if_exists='append', 
    index=False,
    schema='dbo'
)

-1